In [3]:
! ls ../../

block-sparse-featurizer		protein_manifold_inputs.pkl  Untitled.ipynb
Homo_sapiens.GRCh38.pep.all.fa	Untitled1.ipynb


In [19]:
import gc
import torch

# z and atoms are already NumPy arrays, so the BSF model and x no longer
# need to occupy GPU memory while ESMFold is running.
if "model" in globals() and hasattr(model, "cpu"):
    model = model.cpu()

if "x" in globals() and hasattr(x, "cpu"):
    x = x.cpu()

# These variables from the final ESM-2 batch can retain GPU tensors.
for variable_name in (
    "results",
    "token_representations",
    "batch_tokens",
):
    globals().pop(variable_name, None)

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.backends.cuda.matmul.allow_tf32 = True


from transformers import EsmForProteinFolding


fold_device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

fold_model = EsmForProteinFolding.from_pretrained(
    "facebook/esmfold_v1",
    low_cpu_mem_usage=True,
)

fold_model.eval()

if fold_device.type == "cuda":
    # Keep the folding trunk in float32, but use float16 for the large ESM stem.
    fold_model.esm = fold_model.esm.half()

fold_model = fold_model.to(fold_device)

# Reduces the memory used by axial attention for longer proteins.
if hasattr(fold_model, "trunk") and hasattr(
    fold_model.trunk,
    "set_chunk_size",
):
    fold_model.trunk.set_chunk_size(64)

print("ESMFold device:", next(fold_model.parameters()).device)

/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 4498/4498 [00:03<00:00, 1137.44it/s]
[transformers] EsmForProteinFolding LOAD REPORT from: facebook/esmfold_v1
Key                                | Status  | 
-----------------------------------+---------+-
esm.contact_head.regression.bias   | MISSING | 
esm.contact_head.regression.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


ESMFold device: cuda:0


In [4]:
import pickle

with open("../../protein_manifold_inputs.pkl", "rb") as f:
    saved = pickle.load(f)

z = saved["z"]
atoms = saved["atoms"]
ten_proteins = saved["ten_proteins"]
seq_chunk_dict = saved["seq_chunk_dict"]

In [40]:
import numpy as np
import pandas as pd

concept_idx = 95
activation_threshold = 1e-6

# Convert z to NumPy if it is still a PyTorch tensor.
z_np = (
    z.detach().cpu().numpy()
    if hasattr(z, "detach")
    else np.asarray(z)
)

if not 0 <= concept_idx < z_np.shape[1]:
    raise IndexError(
        f"Concept {concept_idx} is out of range. "
        f"Valid concepts are 0 to {z_np.shape[1] - 1}."
    )

# Per-residue activation magnitude for concept 95.
# Shape: (total_number_of_residues,)
concept_activation = np.linalg.norm(
    z_np[:, concept_idx, :],
    axis=-1,
)

rows = []

for protein_id, (start, end) in seq_chunk_dict.items():
    start = int(start)
    end = int(end)

    sequence = ten_proteins[protein_id]
    residue_activation = concept_activation[start:end]

    if len(sequence) != len(residue_activation):
        raise ValueError(
            f"Length mismatch for {protein_id}: "
            f"sequence has {len(sequence)} residues, "
            f"but z slice has {len(residue_activation)}."
        )

    active_mask = residue_activation > activation_threshold

    # Keep only proteins on which the concept activates on at least min_active_residues
    min_active_residues = 0

    if active_mask.sum() < min_active_residues:
        continue

    active_positions = np.flatnonzero(active_mask) + 1  # 1-based positions

    rows.append(
        {
            "protein_id": protein_id,
            "length": len(sequence),

            # Direct analogue of summed squared patch activity.
            "total_activation_energy": float(
                np.square(residue_activation).sum()
            ),

            # Additional useful summaries.
            "total_activation": float(residue_activation.sum()),
            "max_activation": float(residue_activation.max()),
            "mean_activation": float(residue_activation.mean()),
            "mean_active_activation": float(
                residue_activation[active_mask].mean()
            ),
            "active_residue_count": int(active_mask.sum()),
            "active_residue_fraction": float(active_mask.mean()),

            # Location of the strongest residue, using 1-based indexing.
            "strongest_residue_position": int(
                residue_activation.argmax() + 1
            ),
            "strongest_residue_amino_acid": sequence[
                residue_activation.argmax()
            ],

            # All active positions, also 1-based.
            "active_residue_positions": active_positions.tolist(),

            "sequence": sequence,
        }
    )

concept_95_proteins = (
    pd.DataFrame(rows)
    .sort_values(
        "total_activation_energy",
        ascending=False,
    )
    .reset_index(drop=True)
)

concept_95_proteins.insert(
    0,
    "rank",
    np.arange(1, len(concept_95_proteins) + 1),
)

print(
    f"Concept {concept_idx} activates on "
    f"{len(concept_95_proteins)} proteins."
)

display(concept_95_proteins)

Concept 95 activates on 1000 proteins.


/tmp/ipykernel_1762/3303425764.py:68: RuntimeWarning: Mean of empty slice.
  residue_activation[active_mask].mean()
/usr/local/lib/python3.10/dist-packages/numpy/core/_methods.py:192: RuntimeWarning: invalid value encountered in divide
  ret = ret.dtype.type(ret / rcount)


,rank,protein_id,length,total_activation_energy,total_activation,max_activation,mean_activation,mean_active_activation,active_residue_count,active_residue_fraction,strongest_residue_position,strongest_residue_amino_acid,active_residue_positions,sequence
0,1,ENSP00000488778.1 pep scaffold:GRCh38:HSCHR7_2...,124,6786.428711,436.267151,21.585962,3.518283,14.073133,31,0.250000,25,G,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...",MLSPDLPDSAWNTRLLCRVMLCLLGAGSVAAGVIQSPRHLIKEKRE...
1,2,ENSP00000477580.1 pep chromosome:GRCh38:7:1425...,124,6784.914062,436.209961,21.580070,3.517822,14.071289,31,0.250000,25,G,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...",MLSPDLPDSAWNTRLLCRVMLCLLGAGSVAAGVIQSPRHLIKEKRE...
2,3,ENSP00000488756.1 pep scaffold:GRCh38:HSCHR7_2...,114,6776.395508,365.885712,23.614616,3.209524,16.631170,22,0.192982,12,S,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...",MSIGLLCCVAFSLLWASPVNAGVTQTPKFQVLKTGQSMTLQCAQDM...
3,4,ENSP00000374876.2 pep chromosome:GRCh38:7:1423...,114,6776.395508,365.885712,23.614616,3.209524,16.631170,22,0.192982,12,S,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...",MSIGLLCCVAFSLLWASPVNAGVTQTPKFQVLKTGQSMTLQCAQDM...
4,5,ENSP00000488123.1 pep scaffold:GRCh38:HSCHR7_2...,114,6722.448730,368.604858,23.321018,3.233376,16.026300,23,0.201754,12,S,"[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14...",MSIGLLCCAALSLLWAGPVNAGVTQTPKFQVLKTGQSMTLQCAQDM...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,996,ENSP00000516887.1 pep scaffold:GRCh38:HG1369_P...,375,0.000000,0.000000,0.000000,0.000000,NaN,0,0.000000,1,M,[],MEEEIAALVIDNGSGMCKAGFAGDDAPRAVFPSIVGRPRHQGVMVG...
996,997,ENSP00000516864.1 pep scaffold:GRCh38:HG1343_H...,200,0.000000,0.000000,0.000000,0.000000,NaN,0,0.000000,1,M,[],MDNRNTQMYTEGRTKAPGTQPSPGLRITIKRAGVEPIIPGVSKVMF...
997,998,ENSP00000516866.1 pep scaffold:GRCh38:HG1343_H...,200,0.000000,0.000000,0.000000,0.000000,NaN,0,0.000000,1,M,[],MDNRNTQMYTEGRTKAPGTQPSPGLRITIKRAGVEPIIPGVSKVMF...
998,999,ENSP00000329106.3 pep chromosome:GRCh38:Y:2298...,106,0.000000,0.000000,0.000000,0.000000,NaN,0,0.000000,1,M,[],MMTLVPRARTRAGQDHYSHPCPRFSQVLLTEGIMTYCLTKNLSDVN...


In [43]:
concept_95_proteins['protein_id'][20]

'ENSP00000374884.3 pep chromosome:GRCh38:7:142384329:142384841:1 gene:ENSG00000211714.3 transcript:ENST00000390361.3 gene_biotype:TR_V_gene transcript_biotype:TR_V_gene gene_symbol:TRBV7-3 description:T cell receptor beta variable 7-3 [Source:HGNC Symbol;Acc:HGNC:12237]'

In [47]:
import numpy as np
import py3Dmol
from matplotlib.colors import to_hex, to_rgb

# ------------------------------------------------------------
# Choose the concept and protein you want to inspect
# Example protein: concept_95_proteins['protein_id'][945]
# Prefer iloc for positional indexing:
# protein_id = concept_95_proteins["protein_id"].iloc[945]
# ------------------------------------------------------------
concept_idx = 95
protein_id = concept_95_proteins["protein_id"].iloc[500]

activation_threshold = 1e-6
saturation = 1.0
activity_gamma = 0.7
viewer_width = 900
viewer_height = 700

# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------
def manifold_rgb(points, saturation=1.0):
    norm = np.linalg.norm(points, axis=-1, keepdims=True)
    unit = points / np.maximum(norm, 1e-8)
    return np.clip(0.5 + 0.5 * saturation * unit, 0.0, 1.0)

def pdb_center_from_ca(pdb_string):
    coords = []
    for line in pdb_string.splitlines():
        if line.startswith("ATOM  ") and line[12:16].strip() == "CA":
            try:
                coords.append([
                    float(line[30:38]),
                    float(line[38:46]),
                    float(line[46:54]),
                ])
            except ValueError:
                pass

    if len(coords) == 0:
        return {"x": 0.0, "y": 0.0, "z": 0.0}

    center = np.asarray(coords).mean(axis=0)
    return {
        "x": float(center[0]),
        "y": float(center[1]),
        "z": float(center[2]),
    }

# Convert to NumPy if needed
z_np = z.detach().cpu().numpy() if hasattr(z, "detach") else np.asarray(z)
atoms_np = atoms.detach().cpu().numpy() if hasattr(atoms, "detach") else np.asarray(atoms)

if protein_id not in ten_proteins:
    raise KeyError(f"{protein_id!r} not found in ten_proteins")

if protein_id not in seq_chunk_dict:
    raise KeyError(f"{protein_id!r} not found in seq_chunk_dict")

sequence = ten_proteins[protein_id]
start, end = seq_chunk_dict[protein_id]
start, end = int(start), int(end)

if len(sequence) != (end - start):
    raise ValueError(
        f"Length mismatch for {protein_id}: "
        f"sequence has {len(sequence)} residues, slice has {end - start}"
    )

# ------------------------------------------------------------
# 1) Fit the concept manifold using all active residues
# ------------------------------------------------------------
all_heat = np.linalg.norm(z_np[:, concept_idx, :], axis=-1)
active_idx = np.flatnonzero(all_heat > activation_threshold)

if len(active_idx) < 8:
    raise ValueError(
        f"Concept {concept_idx} has too few active residues "
        f"({len(active_idx)}) to fit a manifold."
    )

all_contributions = z_np[active_idx, concept_idx, :] @ atoms_np[concept_idx]   # (n_active, d)
manifold_mean = all_contributions.mean(axis=0)
manifold_components = np.linalg.svd(
    all_contributions - manifold_mean,
    full_matrices=False,
)[2][:3]

# ------------------------------------------------------------
# 2) Compute this protein's residue-level concept activity/colors
# ------------------------------------------------------------
protein_codes = z_np[start:end, concept_idx, :]                  # (L, K)
protein_heat = np.linalg.norm(protein_codes, axis=-1)            # (L,)
protein_contributions = protein_codes @ atoms_np[concept_idx]    # (L, d)
protein_points = (protein_contributions - manifold_mean) @ manifold_components.T  # (L, 3)

protein_rgb = manifold_rgb(protein_points, saturation=saturation)

# Blend weakly active residues toward gray
inactive_rgb = np.asarray(to_rgb("lightgray"))
strength = protein_heat / max(float(protein_heat.max()), 1e-12)
strength = np.clip(strength, 0.0, 1.0) ** activity_gamma
strength[protein_heat <= activation_threshold] = 0.0

display_rgb = (
    inactive_rgb[None, :] * (1.0 - strength[:, None])
    + protein_rgb * strength[:, None]
)

residue_colors = {
    i + 1: to_hex(display_rgb[i])
    for i in range(len(sequence))
}

active_positions = np.flatnonzero(protein_heat > activation_threshold) + 1

# ------------------------------------------------------------
# 3) Fold the protein and display the structure
# Assumes you already loaded ESMFold as `fold_model`
# ------------------------------------------------------------
with torch.inference_mode():
    pdb_string = fold_model.infer_pdb(sequence)

viewer = py3Dmol.view(width=viewer_width, height=viewer_height)
viewer.addModel(pdb_string, "pdb")

# Default all residues to gray
viewer.setStyle({}, {"cartoon": {"color": "lightgray"}})

# Color each residue according to where it lies on the concept manifold
for resi, color in residue_colors.items():
    viewer.setStyle(
        {"resi": str(resi)},
        {"cartoon": {"color": color}},
    )

# Add a label with some summary info
viewer.addLabel(
    (
        f"Protein: {protein_id.split()[0]}\n"
        f"Concept: {concept_idx}\n"
        f"Length: {len(sequence)}\n"
        f"Active residues: {len(active_positions)}\n"
        f"Max activation: {protein_heat.max():.4f}"
    ),
    {
        "position": pdb_center_from_ca(pdb_string),
        "fontColor": "black",
        "backgroundColor": "white",
        "backgroundOpacity": 0.8,
        "fontSize": 12,
        "inFront": True,
    },
)

viewer.setBackgroundColor("white")
viewer.zoomTo()
viewer.show()

print("protein_id:", protein_id)
print("sequence length:", len(sequence))
print("number of active residues:", len(active_positions))
print("active residue positions (1-based):", active_positions.tolist())
print("top 20 residues by activation:")
top_res = np.argsort(-protein_heat)[:20]
for idx in top_res:
    print(
        f"  residue {idx+1:4d}  aa={sequence[idx]}  activation={protein_heat[idx]:.6f}"
    )

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

protein_id: ENSP00000815834.1 pep chromosome:GRCh38:Y:2690988:2741312:1 gene:ENSG00000292348.3 transcript:ENST00001146028.1 gene_biotype:protein_coding transcript_biotype:protein_coding gene_symbol:CD99 description:CD99 molecule (Xg blood group) [Source:HGNC Symbol;Acc:HGNC:7082]
sequence length: 182
number of active residues: 17
active residue positions (1-based): [5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 63, 143]
top 20 residues by activation:
  residue   13  aa=G  activation=10.948199
  residue   14  aa=L  activation=8.281071
  residue   16  aa=G  activation=8.269753
  residue   12  aa=F  activation=7.702593
  residue   11  aa=L  activation=7.515774
  residue   15  aa=L  activation=7.359097
  residue   10  aa=L  activation=7.100819
  residue    9  aa=L  activation=6.245323
  residue   17  aa=V  activation=6.180445
  residue    8  aa=A  activation=5.801851
  residue   18  aa=L  activation=5.108596
  residue   19  aa=V  activation=4.523189
  residue    6  aa=A  activatio